# Experiment 8 - Part 3: The Complete Voice Assistant

**Course:** HAAI302COM - Artificial Intelligence Laboratory

We now join the three links into one object, `VoiceAssistant`: **audio in, audio out**. Then we do what an engineer does with any pipeline: measure the delay of each stage, find the slowest one, and measure how much accuracy the spoken route loses compared with typing the same question.

At the end there is a live-microphone version for your own computer or for Colab.

## 0. Setup

In [1]:
import json, os, re, time, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from faster_whisper import WhisperModel
from IPython.display import Audio, display
import voice_utils as vu

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 170)
os.makedirs("data", exist_ok=True)
ears = {n: WhisperModel(n, device="cpu", compute_type="int8", cpu_threads=8) for n in ["tiny", "base"]}
print("ears loaded:", list(ears))

ears loaded: ['tiny', 'base']


## 1. The assistant, stage by stage

The class has one method per stage and one `respond` method that chains them and times each stage with `time.time()`:

1. `listen`: audio array to text (Whisper). `vad_filter=True` drops non-speech first.
2. `think`: text to reply (Gemini). It keeps the conversation history, so follow-up questions work. The system prompt asks for short spoken sentences.
3. `speak`: reply to an audio file (espeak offline by default, or gTTS). The reply is passed through `clean_for_speech` first.

Constructor arguments are the knobs we will turn later: which Whisper size, which TTS engine, how many past turns to remember.

In [2]:
VOICE_SYS = ("You are a friendly voice assistant. Your reply will be read aloud by a speech engine. Answer in at most three short, "
             "plain sentences. Never use bullet points, numbered lists, headings, asterisks, emojis or links. "
             "Write numbers and units the way a person would say them.")

def clean_for_speech(t):
    t = re.sub(r"```.*?```", " ", t, flags=re.S)
    t = re.sub(r"https?://\S+", " ", t)
    t = re.sub(r"[*_`#>]+", "", t)
    t = re.sub(r"^\s*(?:[-+]|\d+[.)])\s+", "", t, flags=re.M)
    return re.sub(r"\s+", " ", t).strip()

In [3]:
class VoiceAssistant:
    def __init__(self, ear="base", tts="espeak", memory_turns=4, model="gemini-2.5-flash-lite"):
        self.ear, self.tts, self.memory_turns, self.model = ears[ear], tts, memory_turns, model
        self.history = []

    def listen(self, audio):
        text, _, _, _ = vu.transcribe(self.ear, audio, beam_size=1, vad_filter=True)
        return text

    def think(self, text):
        msgs = [{"role": "system", "content": VOICE_SYS}] + self.history[-2 * self.memory_turns:] + [{"role": "user", "content": text}]
        reply = clean_for_speech(vu.chat(msgs, model=self.model)["answer"])
        self.history += [{"role": "user", "content": text}, {"role": "assistant", "content": reply}]
        return reply

    def speak(self, reply, path):
        return vu.tts_espeak(reply, path) if self.tts == "espeak" else vu.tts_gtts(reply, path)

    def respond(self, audio, out_path="data/reply.wav"):
        t = time.time(); heard = self.listen(audio); t_asr = time.time() - t
        t = time.time(); reply = self.think(heard or "I did not catch that."); t_llm = time.time() - t
        t = time.time(); self.speak(reply, out_path); t_tts = time.time() - t
        return dict(heard=heard, reply=reply, audio=out_path, asr=t_asr, llm=t_llm, tts=t_tts, total=t_asr + t_llm + t_tts)

**What to notice:** the stages run one after another, so the delay the user feels is the sum of the three timings, not the largest. `respond` wraps each stage in `time.time()` to record that sum. `memory_turns` keeps only the last few question-answer pairs (the slice `history[-2 * memory_turns:]`), so the prompt does not grow forever.

## 2. A spoken conversation

The "user" is again a synthetic voice (Google TTS, Indian English). Four turns, where the second and third depend on the first ("How tall is it?" has no meaning without the previous turn). The `history` list inside the assistant carries that memory. The LLM answers are cached; the first run of the cell calls Gemini.

In [4]:
DIALOG = ["What is the tallest mountain in the world?", "How tall is it?", "Which country is it in?", "Thank you, that was helpful."]
for i, q in enumerate(DIALOG):
    if not os.path.exists(f"data/d_{i}.mp3"):
        vu.tts_gtts(q, f"data/d_{i}.mp3")

bot = VoiceAssistant(ear="base", tts="espeak")
log = []
for i, q in enumerate(DIALOG):
    r = bot.respond(vu.load_audio(f"data/d_{i}.mp3"), out_path=f"data/reply_{i}.wav")
    log.append(dict(said=q, heard=r["heard"], reply=r["reply"], **{k: r[k] for k in ["asr", "llm", "tts", "total"]}))
turns = pd.DataFrame(log)
turns[["said", "heard", "reply"]]

,said,heard,reply
0,What is the tallest mountain in the world?,What is the tallest mountain in the world?,The tallest mountain in the world is Mount Everest. It is located in the Himalayas on ...
1,How tall is it?,How tall is it?,Mount Everest is twenty-nine thousand thirty-two feet tall. That is eight thousand eig...
2,Which country is it in?,Which country is it in?,Mount Everest is located on the border between Nepal and China. It is part of the Maha...
3,"Thank you, that was helpful.","Thank you, that was helpful.",You are most welcome. I am glad I could help.


**What to notice:** all four turns were heard correctly, and the follow-ups work. 'How tall is it?' gets Mount Everest's height, written out in words for speech (twenty-nine thousand thirty-two feet), and 'Which country is it in?' correctly says Nepal and China. The word 'it' is resolved only because the history is sent with each question.

In [5]:
turns[["asr", "llm", "tts", "total"]].round(2).assign(turn=range(1, 5)).set_index("turn")

,asr,llm,tts,total
turn,,,,
1,2.64,0.0,0.06,2.70
2,2.58,0.0,0.05,2.63
3,2.44,0.0,0.03,2.48
4,2.39,0.0,0.04,2.42


**What to notice:** in this table the ear (base Whisper, about 2.4 to 2.6 seconds per turn) is the largest cost and the espeak mouth is negligible (under 0.1 seconds). The LLM column shows 0.0 because these answers were cached when the notebook was re-run. On the first live run, before caching, the LLM took 1.8, 1.0, 1.0 and 0.9 seconds, so the real total was about 3.3 to 4.5 seconds per turn. A user waits several seconds before hearing anything, which is why the next section looks at cheaper configurations.

In [6]:
display(Audio("data/reply_0.wav"))        # the assistant's first spoken answer

## 3. Experiment: which configuration is fastest?

We vary the ear (tiny or base Whisper) and the mouth (espeak or gTTS) on the same question and compare the total delay. The LLM answer is cached after the first call, so `llm` is nearly zero here; **in real use add the network time of a live call** (0.5 to 2 seconds in notebook 02).

In [7]:
audio0 = vu.load_audio("data/d_0.mp3")
rows = []
for ear in ["tiny", "base"]:
    for tts in ["espeak", "gtts"]:
        b = VoiceAssistant(ear=ear, tts=tts)
        r = b.respond(audio0, out_path=f"data/cfg_{ear}_{tts}." + ("wav" if tts == "espeak" else "mp3"))
        rows.append(dict(ear=ear, mouth=tts, asr=r["asr"], tts=r["tts"], total=r["total"], heard=r["heard"][:40]))
pd.DataFrame(rows).round(2)

,ear,mouth,asr,tts,total,heard
0,tiny,espeak,1.17,0.03,1.20,which is the tallest mountain in the wor
1,tiny,gtts,1.13,1.78,2.92,which is the tallest mountain in the wor
2,base,espeak,2.33,0.04,2.37,What is the tallest mountain in the worl
3,base,gtts,2.22,1.73,3.95,What is the tallest mountain in the worl


**What to notice:** the ear and the mouth both move the total. The tiny ear needs about 1.1 to 1.2 seconds against 2.2 to 2.3 for base; gTTS adds 1.6 to 1.8 seconds against 0.03 to 0.05 for espeak. The total ranges from 1.2 seconds (tiny and espeak) to 3.95 seconds (base and gTTS), more than three times, before adding the LLM. The fast choice sounds robotic and hears less well, and the natural choice is slower; a product must choose where to sit between them.

## 4. How much accuracy does the voice route cost?

Typing "What is the capital of Japan?" and saying it are not the same input. We take 10 factual questions with a known answer keyword, and ask them four ways:

- **typed**: the exact text goes to the LLM (the upper bound; the ear is not involved).
- **spoken, base ear, clean audio**.
- **spoken, tiny ear, noisy audio (5 dB)**: a cheap ear in a noisy room.
- **same, with `language="en"` forced**: we tell the ear the language instead of letting it guess.

The reply counts as correct if it contains any accepted keyword. The gap between "typed" and the spoken rows is the **cost of the ear**.

In [8]:
QA = [("What is the capital of Japan?", ["tokyo"]), ("Who painted the Mona Lisa?", ["leonardo", "da vinci"]),
      ("What is the chemical symbol for gold?", ["au"]), ("How many continents are there?", ["seven", "7"]),
      ("What is the largest ocean on Earth?", ["pacific"]), ("Who discovered penicillin?", ["fleming"]),
      ("What is the square root of one hundred forty four?", ["twelve", "12"]),
      ("In which year did India become independent?", ["1947", "nineteen forty seven", "nineteen forty-seven"]),
      ("What gas do plants absorb from the air?", ["carbon dioxide", "co2"]), ("What is the largest planet in the solar system?", ["jupiter"])]
for i, (q, _) in enumerate(QA):
    if not os.path.exists(f"data/qa_{i}.mp3"):
        vu.tts_gtts(q, f"data/qa_{i}.mp3")

def ask_once(text):
    # a fresh assistant per question so that memory from earlier questions cannot help
    return clean_for_speech(vu.chat([{"role": "system", "content": VOICE_SYS}, {"role": "user", "content": text}])["answer"])

def correct(reply, keys):
    return any(k in vu.norm_text(reply) for k in keys)

res = []
for i, (q, keys) in enumerate(QA):
    a = vu.load_audio(f"data/qa_{i}.mp3")
    h_base = vu.transcribe(ears["base"], a, beam_size=1)[0]
    h_tiny = vu.transcribe(ears["tiny"], vu.add_noise(a, 5), beam_size=1)[0]
    h_en = vu.transcribe(ears["tiny"], vu.add_noise(a, 5), beam_size=1, language="en")[0]
    res.append(dict(question=q, typed=correct(ask_once(q), keys),
                    base_clean=correct(ask_once(h_base), keys), tiny_5dB=correct(ask_once(h_tiny or "silence"), keys),
                    tiny_5dB_en=correct(ask_once(h_en or "silence"), keys),
                    heard_base=h_base, heard_tiny=h_tiny, heard_en=h_en))
R = pd.DataFrame(res)
R[["typed", "base_clean", "tiny_5dB", "tiny_5dB_en"]].mean().round(2).to_frame("accuracy")

,accuracy
typed,1.0
base_clean,1.0
tiny_5dB,0.4
tiny_5dB_en,0.5


**What to notice:** with a good ear the spoken route loses nothing here: typed, and spoken with the base ear on clean audio, both score 10 of 10. The cheap tiny ear in 5 dB noise falls to 4 of 10, and 5 of 10 with English forced. So the whole accuracy loss in this pipeline comes from the ear, not the LLM. Ten questions is a small test: each question is 10 percentage points, so read the table as a clear pattern, not a precise measurement.

In [9]:
bad = R[~R.typed | ~R.base_clean | ~R.tiny_5dB | ~R.tiny_5dB_en][["question", "tiny_5dB", "tiny_5dB_en", "heard_tiny", "heard_en"]]
bad.assign(question=bad.question.str[:34], heard_tiny=bad.heard_tiny.str[:32], heard_en=bad.heard_en.str[:32])

,question,tiny_5dB,tiny_5dB_en,heard_tiny,heard_en
1,Who painted the Mona Lisa?,False,False,후에 인증도 많아입고,Who can take the moment or what?
4,What is the largest ocean on Earth,False,False,which is the largest pollution o,which is the largest pollution o
5,Who discovered penicillin?,False,False,Şimdi tavır canıçılın,Who discovered the machine?
6,What is the square root of one hun,False,False,which is the 4.144,which is the fuel unit of 144
7,In which year did India become ind,False,True,اور پھنچہ قاملات دنیار میجن کی د,In which year did India become i
8,What gas do plants absorb from the,False,False,What does he plan for job from t,What does he plan for job from t


**What to notice:** the six failures show two different problems. Three transcripts were in the wrong script (Korean, Turkish, Urdu), and forcing English fixed only the India independence question. For the others, forcing English turned foreign-script junk into plausible English junk ('Who discovered the machine?' for penicillin, 'fuel unit of 144' for the square root of 144), which the LLM then answers wrongly. The remaining errors are acoustic: the tiny model cannot hear well through that much noise. The fixes are a bigger ear, noise suppression or a better microphone, not a language flag.

## 5. Going live with a microphone (run on your own computer or Colab)

The notebook cannot record in this environment, so the code below is only **defined**, not run. On your laptop install `pip install sounddevice` (Linux also needs the system library `libportaudio2`). In Colab, recording from the browser needs a small JavaScript helper or a Gradio microphone widget (`gr.Audio(sources=["microphone"])`) that returns a file path, which you pass to `load_audio` and `respond`.

The loop is: record a few seconds, call `respond`, play the reply, repeat. Stop it with the interrupt button.

In [10]:
def record(seconds=5, sr=vu.SR):
    import sounddevice as sd                      # imported here so the notebook runs without it
    print(f"Recording {seconds} s ...")
    x = sd.rec(int(seconds * sr), samplerate=sr, channels=1, dtype="float32"); sd.wait()
    return x[:, 0]

def live_loop(bot, seconds=5):
    while True:
        r = bot.respond(record(seconds))
        print("You said :", r["heard"]); print("Assistant:", r["reply"], f"({r['total']:.1f} s)")
        display(Audio(r["audio"], autoplay=True))

# live_loop(VoiceAssistant(ear="base", tts="gtts"))     # uncomment on your own machine
print("live_loop defined; uncomment the last line to talk to the assistant")

live_loop defined; uncomment the last line to talk to the assistant


## 6. Recap and exercises

- The assistant is three functions in a chain; `respond` times each one.
- Latency is the sum of ear, brain and mouth; the biggest term is usually the LLM or a slow TTS.
- Accuracy through voice is bounded by the ear; evaluate typed versus spoken input to measure the loss.
- Conversation memory is the resent history; limit its length to bound the cost.

**Try yourself:** (1) set `memory_turns=0` and rerun the dialogue to see "How tall is it?" fail; (2) record yourself with `live_loop`; (3) add a second language with `language="hi"` in the Whisper call and a Hindi system prompt.

This completes Experiment 8.